# Task 2 — Evasion‑Based Clarity Classification

Fine‑tuning di **Llama 3.1 8B‑Instruct** (4‑bit QLoRA + DoRA) per classificare la tecnica di evasione politica tra 10 sotto‑categorie, poi mappata alle 3 macro‑categorie di chiarezza.

In [1]:
# ============================================================
# CELLA 1 — Setup & Installazione
# ============================================================

# Installazione dipendenze (decommentare su Colab)
# !pip install -q -U transformers peft trl bitsandbytes datasets accelerate

import os, sys, json, gc
import torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from huggingface_hub import login
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import load_from_disk
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix

# Configurazione dei percorsi e rilevamento ambiente (Colab / Locale)
try:
    import google.colab
    from google.colab import drive, userdata
    drive.mount('/content/drive', force_remount=True)

    # Percorsi base su Google Drive
    BASE_DIR = "/content/drive/MyDrive/progettoLLM"
    REPO_DIR = os.path.join(BASE_DIR, "CLARITY")
    hf_cache_dir = os.path.join(BASE_DIR, "hf_cache")
    
    os.makedirs(hf_cache_dir, exist_ok=True)
    os.environ["HF_HOME"] = hf_cache_dir
    
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    # Login HF tramite Colab Secrets
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    login(token=hf_token)
    print(f"Ambiente Google Colab configurato. Cache: {hf_cache_dir}")

except ImportError:
    # Percorsi base in locale
    BASE_DIR = ".."
    REPO_DIR = os.path.join(BASE_DIR, "CLARITY")
    
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    # Lettura del token da file .env locale
    env_path = os.path.join(REPO_DIR, ".env")
    if not os.path.exists(env_path):
        env_path = ".env"
        
    if os.path.exists(env_path):
        with open(env_path) as f:
            for line in f:
                if line.startswith("HF_TOKEN="):
                    hf_token = line.split("=", 1)[1].strip().strip("'\"")
                    os.environ['HF_TOKEN'] = hf_token
                    login(token=hf_token)
                    break
    print("Ambiente locale rilevato.")

print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Ambiente locale rilevato.
Device: NVIDIA GeForce RTX 2080 Ti


In [2]:
# ============================================================
# CELLA 2 — Configurazione e Variabili Globali
# ============================================================

# Flag di controllo
ESEGUI_TRAINING = True   # True = addestra da zero, False = carica checkpoint

# Modello e percorsi
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

# 1. Cartella Base per i risultati del Task 2
TASK2_DIR = os.path.join(BASE_DIR, "risultati_task2")

# 2. Percorsi specifici costruiti dinamicamente
OUTPUT_DIR = os.path.join(TASK2_DIR, "checkpoints")
PATH_MODELLO_SALVATO = os.path.join(TASK2_DIR, "modello_finale")
RISULTATI_DIR = os.path.join(TASK2_DIR, "valutazione")

# Precisione: bfloat16 se supportato, altrimenti float16
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"Compute dtype: {COMPUTE_DTYPE}")

# System prompt con le 10 categorie del Task 2
SYSTEM_PROMPT_TASK2 = """Sei un esperto analista di comunicazione politica. Il tuo compito è classificare la risposta del politico alla domanda scegliendo ESATTAMENTE UNA delle seguenti 9 categorie:

1. 'Explicit'
2. 'Declining to answer'
3. 'Claims ignorance'
4. 'Dodging'
5. 'Deflection'
6. 'General'
7. 'Implicit'
8. 'Partial/half-answer'
9. 'Clarification'

Esempi di classificazione (Few-Shot):
Q: "Appoggerà la nuova legge sulle tasse?"
A: "Non sono a conoscenza dei dettagli della legge." -> {"categoria": "Claims ignorance"}

Q: "Cosa pensa dello scandalo del suo partito?"
A: "Dovreste guardare a cosa ha fatto l'opposizione l'anno scorso!" -> {"categoria": "Deflection"}

Q: "Quali misure prenderete per il clima?"
A: "Il clima è importante, faremo del nostro meglio per tutti." -> {"categoria": "General"}

Rispondi ESCLUSIVAMENTE fornendo un oggetto JSON valido contenente la singola chiave "categoria". Non aggiungere formattazione markdown (come ```json), né preamboli o spiegazioni.
"""
print("Configurazione caricata.")

Compute dtype: torch.bfloat16
Configurazione caricata.


In [3]:
# ============================================================
# CELLA 3 — Preparazione Dataset
# ============================================================

# Caricamento tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Caricamento split train e test utilizzando percorsi assoluti
TRAIN_DATA_PATH = os.path.join(REPO_DIR, "dataset", "QEvasion", "train")
TEST_DATA_PATH = os.path.join(REPO_DIR, "dataset", "QEvasion", "test")

train_dataset = load_from_disk(TRAIN_DATA_PATH)
test_dataset  = load_from_disk(TEST_DATA_PATH)
print(f"Train: {len(train_dataset)} esempi | Test: {len(test_dataset)} esempi")

def format_prompt_task2(example):
    """Formatta un esempio nel template chat di Llama 3.1 per il Task 2."""
    domanda  = example.get('interview_question', '')
    risposta = example.get('interview_answer', '')
    label    = example.get('evasion_label', 'Explicit')

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_TASK2},
        {"role": "user",   "content": f"Domanda: {domanda}\nRisposta del politico: {risposta}"},
        {"role": "assistant", "content": str(label)},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

# Applica la formattazione solo al train set (il test usa la funzione di inferenza)
formatted_train = train_dataset.map(format_prompt_task2, remove_columns=train_dataset.column_names)

print("\n--- Esempio di prompt formattato ---")
print(formatted_train[0]["text"][:500] + "\n...")

Train: 3448 esempi | Test: 308 esempi

--- Esempio di prompt formattato ---
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

Sei un esperto analista di comunicazione politica. Il tuo compito è classificare la risposta del politico alla domanda scegliendo ESATTAMENTE UNA delle seguenti 9 categorie:

1. 'Explicit'
2. 'Declining to answer'
3. 'Claims ignorance'
4. 'Dodging'
5. 'Deflection'
6. 'General'
7. 'Implicit'
8. 'Partial/half-answer'
9. 'Clarification'

Esempi di classificazione (Few-Shot):
Q
...


In [4]:
# ============================================================
# CELLA 4 — Logica del Modello (Training vs Loading)
# ============================================================

# Pulizia VRAM per evitare OOM
torch.cuda.empty_cache()
gc.collect()

# Configurazione quantizzazione 4-bit comune
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

if ESEGUI_TRAINING:
    # ── TRAINING ──────────────────────────────────────────────
    print("Modalità TRAINING attiva.")

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, device_map="auto",
        quantization_config=bnb_config, torch_dtype=COMPUTE_DTYPE,
    )
    model = prepare_model_for_kbit_training(model)
    
    # [NEW] 1. Abilita il Gradient Checkpointing per risparmiare VRAM
    model.gradient_checkpointing_enable()

    # Configurazione DoRA sui layer di attenzione
    peft_config = LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM", use_dora=True,
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    training_args = SFTConfig(             # [CHANGED] Usa SFTConfig invece di TrainingArguments
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=1,     
        gradient_accumulation_steps=8,     
        optim="paged_adamw_32bit",
        save_strategy="epoch",
        logging_steps=10,
        learning_rate=2e-4,
        weight_decay=0.001,
        fp16=(COMPUTE_DTYPE == torch.float16),
        bf16=(COMPUTE_DTYPE == torch.bfloat16),
        max_grad_norm=0.3,
        num_train_epochs=3,
        warmup_steps=100,
        group_by_length=True,
        lr_scheduler_type="cosine",
        report_to="none",
        max_length=512,                    # [NEW] Spostato qui e rinominato in max_length
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=formatted_train,
        args=training_args,
        peft_config=None,          
        processing_class=tokenizer,
        # max_seq_length rimosso da qui!
    )

    print("🚀 Inizio addestramento...")
    trainer.train()

    # Salvataggio adattatori + tokenizer su Drive
    trainer.model.save_pretrained(PATH_MODELLO_SALVATO)
    tokenizer.save_pretrained(PATH_MODELLO_SALVATO)
    print(f"Modello salvato in: {PATH_MODELLO_SALVATO}")

else:
    # ── CARICAMENTO DA CHECKPOINT ─────────────────────────────
    print(f"Caricamento modello base ({MODEL_ID}) in 4-bit...")

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, device_map="auto",
        quantization_config=bnb_config, torch_dtype=COMPUTE_DTYPE,
    )

    print(f"Applicazione adattatori LoRA da: {PATH_MODELLO_SALVATO}")
    model = PeftModel.from_pretrained(
        base_model, PATH_MODELLO_SALVATO,
        low_cpu_mem_usage=True, device_map="auto",
    )

    # Sovrascriviamo il tokenizer con quello salvato col checkpoint
    tokenizer = AutoTokenizer.from_pretrained(PATH_MODELLO_SALVATO)
    print("Modello e adattatori caricati.")

model.eval()
print(f"Modello in eval mode. Device: {model.device}")

Modalità TRAINING attiva.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 13,959,168 || all params: 8,044,220,416 || trainable%: 0.1735


Truncating train dataset:   0%|          | 0/3448 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


🚀 Inizio addestramento...


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,2.581600
20,2.360000
30,1.917500


KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELLA 5 — Funzioni di Supporto (Inference JSON & Mapping)
# ============================================================
import json
import torch

def predici_evasione(example):
    """
    Genera la predizione della tecnica di evasione in formato JSON 
    e la mappa automaticamente alla macro-categoria di chiarezza.
    Restituisce: (tecnica_predetta, macro_clarity, raw_output)
    """
    domanda  = example.get('interview_question', '')
    risposta = example.get('interview_answer', '')

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_TASK2},
        {"role": "user",   "content": f"Domanda: {domanda}\nRisposta: {risposta}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=30,  # Spazio sufficiente per generare {"categoria": "Partial/half-answer"}
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Estrazione della stringa generata dall'LLM
    generated_ids = outputs[0][inputs['input_ids'].shape[-1]:]
    raw_output = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    
    # Pulizia di eventuale markdown residuo (es. ```json ... ```) che l'LLM potrebbe aggiungere
    clean_json_str = raw_output.replace("```json", "").replace("```", "").strip()
    
    # Valore di Default/Fallback se l'LLM allucina la struttura JSON
    tecnica_predetta = "Dodging" 
    
    try:
        data = json.loads(clean_json_str)
        tecnica_predetta = data.get("categoria", "Dodging")
    except json.JSONDecodeError:
        pass # Se il JSON è rotto, tiene il fallback ("Dodging")
        
    # Mapping diretto alle 3 classi (Solo le 9 categorie ufficiali del paper)
    MAPPING_9_CLASSI = {
        "Explicit": "Clear Reply",
        "Declining to answer": "Clear Non-Reply",
        "Claims ignorance": "Clear Non-Reply",
        "Dodging": "Ambivalent",
        "Deflection": "Ambivalent",
        "General": "Ambivalent",
        "Implicit": "Ambivalent",
        "Partial/half-answer": "Ambivalent",
        "Clarification": "Ambivalent"
    }
    
    # Se la tecnica predetta dal JSON non è tra le 9 valide, forza su Ambivalent
    macro_clarity = MAPPING_9_CLASSI.get(tecnica_predetta, "Ambivalent")
    
    return tecnica_predetta, macro_clarity, raw_output

# ============================================================
# Sanity check rapido
# ============================================================
print("--- Sanity check (3 esempi) ---")
for i in range(min(3, len(test_dataset))):
    es = test_dataset[i]
    tecnica, clarity, raw = predici_evasione(es)
    vero = str(es.get('clarity_label', '')).strip()
    print(f"[{i+1}] Raw: {raw:40s} \n    → Tecnica: {tecnica:22s} → Clarity: {clarity:15s} (Vera: {vero})\n")

In [ ]:
# ============================================================
# CELLA 6 — Valutazione e Grafici
# ============================================================

y_true_clarity = []
y_pred_clarity = []
y_pred_raw_evasion = []

print(f"Valutazione su {len(test_dataset)} esempi...")

for i in tqdm(range(len(test_dataset))):
    esempio = test_dataset[i]

    # Etichetta reale (macro-categoria)
    y_true_clarity.append(str(esempio.get('clarity_label', '')).strip())

    # Predizione → normalizzazione → mapping
    tecnica, clarity, raw = predici_evasione(esempio)
    y_pred_raw_evasion.append(tecnica)
    y_pred_clarity.append(clarity)

# Classification Report (3 cifre decimali)
LABELS = ["Ambivalent", "Clear Reply", "Clear Non-Reply"]

print("\n" + "=" * 60)
print("REPORT FINALE — TASK 2 (EVASION → CLARITY)")
print("=" * 60)
print(classification_report(y_true_clarity, y_pred_clarity, labels=LABELS, digits=3))

# Matrice di Confusione
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_true_clarity, y_pred_clarity, labels=LABELS)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABELS, yticklabels=LABELS)
plt.xlabel('Predizioni (via Mapping Task 2)')
plt.ylabel('Etichette Reali')
plt.title('Matrice di Confusione — Strategia Evasion-based (Task 2)')
plt.tight_layout()

# Salvataggio Grafico
os.makedirs(RISULTATI_DIR, exist_ok=True)
percorso_grafico = os.path.join(RISULTATI_DIR, "matrice_confusione.png")
plt.savefig(percorso_grafico, bbox_inches='tight', dpi=300)
plt.show()

# Distribuzione delle tecniche di evasione predette
df_raw = pd.Series(y_pred_raw_evasion).value_counts().reset_index()
df_raw.columns = ['Tecnica Task 2', 'Conteggio']

print("\n--- Distribuzione predizioni grezze (Task 2) ---")
print(df_raw)

# Salvataggio Predizioni in CSV
print(f"\nSalvataggio risultati in corso nella directory: {RISULTATI_DIR}...")
df_risultati = pd.DataFrame({
    'Domanda': [ex.get('interview_question', '') for ex in test_dataset],
    'Risposta_Politico': [ex.get('interview_answer', '') for ex in test_dataset],
    'Vero_Task1_Clarity': y_true_clarity,
    'Vero_Task2_Evasion': [str(ex.get('evasion_label', '')) for ex in test_dataset],
    'Predetto_Task2_Evasion': y_pred_raw_evasion,
    'Predetto_Task1_Clarity': y_pred_clarity
})

percorso_csv = os.path.join(RISULTATI_DIR, "predizioni_test_set.csv")
df_risultati.to_csv(percorso_csv, index=False)
print(f"CSV salvato con successo: {percorso_csv}")